# Test-set perplexity + model complexity: Mamba-2 vs GPT-2 vs BERT

**Why this notebook exists:** the perplexity printed during CPT was *validation* perplexity, computed on 512 blocks held out from the *training* CSV. The dev/test documents were excluded from CPT, so this notebook evaluates all three models on those untouched **test** documents.

**What it reports**
1. Test loss / perplexity for each model, before CPT (base checkpoint) and after CPT (your saved `final/` folder).
2. Bits-per-byte (BPB) for the causal models, and an optional pseudo-log-likelihood (PLL) BPB for BERT. BPB is normalised by the original text bytes, so it is comparable across different tokenizers.
3. Parameters, model size (MiB), forward FLOPs, and estimated CPT training FLOPs.

**Read this before comparing numbers**
- GPT-2 / Mamba-2 perplexity is per-token over their own tokenizers (BPE), so token-level perplexities are only roughly comparable. Use BPB for a fair comparison.
- BERT's MLM perplexity is *pseudo*-perplexity over 15% masked tokens. It is **not** comparable to causal perplexity. Use the PLL rows (all tokens masked one at a time) for a closer comparison, and keep in mind BERT-uncased lowercases text, which makes its task slightly easier.
- Evaluation uses a seeded random sample of the test documents (same documents for all models), and at most `MAX_EVAL_BLOCKS` blocks per model. Set both to `None`/`1.0` for a full-test run.

In [1]:
import os, json, math, time, zipfile, hashlib, pickle, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.flop_counter import FlopCounterMode
from transformers import AutoTokenizer, AutoModelForMaskedLM, GPT2LMHeadModel, Mamba2ForCausalLM

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

DATASET_DIR = Path("Dataset")
RESULTS_DIR = Path("./comparison_results")
RESULTS_DIR.mkdir(exist_ok=True)

TEST_ZIPS = [
    ("binary_test.zip",  "binary_test/CJPE_ext_SCI_HCs_Tribunals_daily_orders_test.csv"),
    ("ternary_test.zip", "ternary_test/CJPE_ext_SCI_HCs_tribunals_dailyorder_test_wo_RoD_ternary.csv"),
]
TEST_FRACTION = 0.02       # fraction of unique test docs to sample (1.0 = all)
MAX_TEST_DOCS = 5000       # cap on sampled docs (None = no cap)
MAX_EVAL_BLOCKS = 2000     # blocks evaluated per model (None = all); 2000 blocks ~ 2M tokens
INCLUDE_BASELINES = True   # also evaluate the pre-CPT base checkpoints
PLL_BLOCKS = 100           # BERT pseudo-log-likelihood on this many blocks (0 = skip)
MLM_PROB = 0.15
USE_BF16 = True
SEED = 42

MODELS = [
    dict(name="Mamba-2 130M", kind="causal", cls=Mamba2ForCausalLM,
         base="AntonV/mamba2-130m-hf", cpt="mamba2-130m-cpt-full/final", block=1024, bs=4),
    dict(name="GPT-2 124M", kind="causal", cls=GPT2LMHeadModel,
         base="gpt2", cpt="gpt2-124m-cpt-full/final", block=1024, bs=4),
    dict(name="BERT-base-uncased", kind="mlm", cls=AutoModelForMaskedLM,
         base="bert-base-uncased", cpt="bert-base-cpt-full/final", block=510, bs=16),
]

W1120 15:17:06.613000 33956 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
c:\Users\HP\anaconda3\envs\phd_pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA RTX 4000 Ada Generation


In [2]:
def doc_key(t):
    return hashlib.md5(" ".join(t.lower().split()).encode("utf-8")).digest()

def iter_chunks(zip_name, csv_name, chunksize=2000):
    with zipfile.ZipFile(DATASET_DIR / zip_name) as z, z.open(csv_name) as f:
        for chunk in pd.read_csv(f, usecols=["filename", "text"], chunksize=chunksize):
            chunk = chunk.dropna(subset=["text"])
            chunk["text"] = chunk["text"].astype(str).str.strip()
            yield chunk[chunk["text"] != ""]

TEXTS_PATH = RESULTS_DIR / "test_texts.pkl"     # delete this file to resample
if TEXTS_PATH.exists():
    with open(TEXTS_PATH, "rb") as f:
        texts, n_unique = pickle.load(f)
else:
    rng = np.random.default_rng(SEED)
    seen, texts, n_unique = set(), [], 0
    for zip_name, csv_name in TEST_ZIPS:
        for chunk in iter_chunks(zip_name, csv_name):
            for t in chunk["text"]:
                k = doc_key(t)
                if k in seen:
                    continue
                seen.add(k)
                n_unique += 1
                if rng.random() < TEST_FRACTION:
                    texts.append(t)
        print(f"{zip_name}: unique docs so far {n_unique:,} | sampled {len(texts):,}")
    rng.shuffle(texts)      # random order, so truncating to MAX_EVAL_BLOCKS stays unbiased
    if MAX_TEST_DOCS is not None:
        texts = texts[:MAX_TEST_DOCS]
    with open(TEXTS_PATH, "wb") as f:
        pickle.dump((texts, n_unique), f)

TEST_BYTES = sum(len(t.encode("utf-8")) for t in texts)
print(f"Unique test docs: {n_unique:,} | evaluated sample: {len(texts):,} docs | {TEST_BYTES/1e6:.1f} MB of text")

binary_test.zip: unique docs so far 110,677 | sampled 2,203
ternary_test.zip: unique docs so far 220,757 | sampled 4,363
Unique test docs: 220,757 | evaluated sample: 4,363 docs | 51.6 MB of text


In [3]:
autocast = lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16)

def build_stream(tokenizer, texts, sep_id, batch=256):
    parts = []
    for i in range(0, len(texts), batch):
        enc = tokenizer(texts[i:i + batch], add_special_tokens=False, truncation=False)["input_ids"]
        parts.extend(np.asarray(ids + [sep_id], dtype=np.int64) for ids in enc)
    return np.concatenate(parts)

def to_blocks(stream, block, max_blocks):
    n = len(stream) // block
    if max_blocks is not None:
        n = min(n, max_blocks)
    return torch.from_numpy(stream[: n * block].reshape(n, block)), len(stream)

@torch.no_grad()
def eval_causal(model, blocks, bs):
    """Returns (sum of NLL in nats, number of predicted tokens)."""
    model.eval()
    nll, ntok = 0.0, 0
    for i in range(0, len(blocks), bs):
        x = blocks[i:i + bs].to(device)
        with autocast():
            logits = model(input_ids=x, use_cache=False).logits
        lg = logits[:, :-1].float()
        nll += F.cross_entropy(lg.reshape(-1, lg.size(-1)), x[:, 1:].reshape(-1), reduction="sum").item()
        ntok += x[:, 1:].numel()
        del logits, lg
    return nll, ntok

def mask_batch(x, tok, gen):
    """Standard BERT masking (15%, 80/10/10) on (B,510) blocks -> (B,512) inputs and labels."""
    CLS, SEP, MASK, V = tok.cls_token_id, tok.sep_token_id, tok.mask_token_id, len(tok)
    B = x.size(0)
    x = torch.cat([torch.full((B, 1), CLS), x, torch.full((B, 1), SEP)], dim=1)
    labels = x.clone()
    special = (x == CLS) | (x == SEP)
    prob = torch.full(x.shape, MLM_PROB)
    prob[special] = 0.0
    m = torch.bernoulli(prob, generator=gen).bool()
    labels[~m] = -100
    r = torch.rand(x.shape, generator=gen)
    inp = x.clone()
    inp[m & (r < 0.8)] = MASK
    rnd = m & (r >= 0.8) & (r < 0.9)
    inp[rnd] = torch.randint(1000, V, x.shape, generator=gen)[rnd]
    return inp, labels

@torch.no_grad()
def eval_mlm(model, blocks, tok, bs):
    """Fixed-seed MLM loss over masked positions -> (sum NLL, number of masked tokens)."""
    model.eval()
    gen = torch.Generator().manual_seed(0)
    nll, n = 0.0, 0
    for i in range(0, len(blocks), bs):
        inp, lab = mask_batch(blocks[i:i + bs], tok, gen)
        inp, lab = inp.to(device), lab.to(device)
        with autocast():
            logits = model(input_ids=inp).logits
        sel = lab != -100
        nll += F.cross_entropy(logits[sel].float(), lab[sel], reduction="sum").item()
        n += int(sel.sum())
        del logits
    return nll, n

@torch.no_grad()
def bert_pll(model, blocks, tok, n_blocks, bs=64):
    """Pseudo-log-likelihood: mask each token in turn and score the true token."""
    model.eval()
    CLS, SEP, MASK = tok.cls_token_id, tok.sep_token_id, tok.mask_token_id
    nll, n = 0.0, 0
    for b in range(min(n_blocks, len(blocks))):
        x = torch.cat([torch.tensor([CLS]), blocks[b], torch.tensor([SEP])]).to(device)
        L = x.numel()
        for s in range(1, L - 1, bs):
            pos = torch.arange(s, min(s + bs, L - 1), device=device)
            ar = torch.arange(len(pos), device=device)
            inp = x.unsqueeze(0).repeat(len(pos), 1)
            inp[ar, pos] = MASK
            with autocast():
                h = model.bert(input_ids=inp).last_hidden_state
                logits = model.cls(h[ar, pos])
            lp = F.log_softmax(logits.float(), dim=-1)
            nll += -lp[ar, x[pos]].sum().item()
            n += len(pos)
    return nll, n

In [4]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())          # tied weights counted once
    emb = model.get_input_embeddings().weight.numel()
    return total, emb

def fwd_flops(model, seq_len, causal):
    """Forward FLOPs (2*MACs) for one sequence, via torch FlopCounterMode (matmul/conv/attention ops)."""
    model.eval()
    x = torch.randint(5, 1000, (1, seq_len), device=device)
    kwargs = dict(input_ids=x)
    if causal:
        kwargs["use_cache"] = False
    try:
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            model(**kwargs)
        return fc.get_total_flops()
    except Exception as e:
        print(f"   FLOP counting failed at seq_len={seq_len}: {type(e).__name__}: {e}")
        return float("nan")

def complexity(model, cfg):
    causal = cfg["kind"] == "causal"
    native = cfg["block"] + (0 if causal else 2)       # BERT blocks get [CLS]/[SEP]
    total, emb = count_params(model)
    cpt_dir = Path(cfg["cpt"])
    st = cpt_dir / "model.safetensors"
    row = {
        "Model": cfg["name"],
        "Params (M)": total / 1e6,
        "Embedding params (M)": emb / 1e6,
        "Non-embedding params (M)": (total - emb) / 1e6,
        "Size fp32 (MiB)": total * 4 / 2**20,
        "Size bf16 (MiB)": total * 2 / 2**20,
        "Size on disk, CPT ckpt (MiB)": st.stat().st_size / 2**20 if st.exists() else float("nan"),
    }
    for L in (512, 1024):
        if not causal and L > 512:
            row[f"Fwd GFLOPs @{L} tok"] = float("nan")       # BERT cannot go beyond 512
            row[f"GFLOPs/token @{L}"] = float("nan")
            continue
        fl = fwd_flops(model, L, causal)
        row[f"Fwd GFLOPs @{L} tok"] = fl / 1e9
        row[f"GFLOPs/token @{L}"] = fl / L / 1e9
    # training FLOPs ~ 3x forward (fwd + bwd) over one epoch of the CPT token stream
    tj = cpt_dir.parent / "tokens_all.json"
    fl_native = fwd_flops(model, native, causal)
    if tj.exists():
        total_tokens = json.loads(tj.read_text())["total_tokens"]
        n_seqs = total_tokens // cfg["block"]
        row["CPT tokens seen (M)"] = n_seqs * cfg["block"] / 1e6
        row["CPT train FLOPs (x1e15)"] = 3 * fl_native * n_seqs / 1e15
    else:
        row["CPT tokens seen (M)"] = float("nan")
        row["CPT train FLOPs (x1e15)"] = float("nan")
    return row

In [5]:
results, complexity_rows = [], []

for cfg in MODELS:
    cpt_dir = Path(cfg["cpt"])
    stages = []
    if INCLUDE_BASELINES:
        stages.append(("pre-CPT", cfg["base"]))
    if cpt_dir.exists():
        stages.append(("post-CPT", str(cpt_dir)))
    else:
        print(f"[skip post-CPT] {cpt_dir} not found for {cfg['name']}")
    if not stages:
        continue

    print(f"\n=== {cfg['name']} ===")
    causal = cfg["kind"] == "causal"
    tok = AutoTokenizer.from_pretrained(cfg["base"])
    tok.model_max_length = int(1e9)
    sep_id = tok.eos_token_id if causal else tok.sep_token_id
    stream = build_stream(tok, texts, sep_id)
    blocks, stream_len = to_blocks(stream, cfg["block"], MAX_EVAL_BLOCKS)
    frac = len(blocks) * cfg["block"] / stream_len          # share of the test text actually scored
    bytes_cov = TEST_BYTES * frac
    print(f"tokens in stream {stream_len:,} | blocks evaluated {len(blocks):,} ({frac:.1%} of the sampled text)")

    first = True
    for stage, path in stages:
        model = cfg["cls"].from_pretrained(path, dtype=torch.float32).to(device)
        torch.cuda.synchronize(); t0 = time.time()
        if causal:
            nll, n = eval_causal(model, blocks, cfg["bs"])
            dt = time.time() - t0
            results.append(dict(Model=cfg["name"], Stage=stage, Objective="causal LM",
                                Loss=nll / n, PPL=math.exp(nll / n),
                                BPB=nll / math.log(2) / bytes_cov, Tokens=n, Seconds=dt))
        else:
            nll, n = eval_mlm(model, blocks, tok, cfg["bs"])
            dt = time.time() - t0
            results.append(dict(Model=cfg["name"], Stage=stage, Objective="MLM (15% masked)",
                                Loss=nll / n, PPL=math.exp(nll / n),
                                BPB=float("nan"), Tokens=n, Seconds=dt))
            if PLL_BLOCKS:
                t1 = time.time()
                pnll, pn = bert_pll(model, blocks, tok, PLL_BLOCKS)
                pfrac = min(PLL_BLOCKS, len(blocks)) * cfg["block"] / stream_len
                results.append(dict(Model=cfg["name"], Stage=stage, Objective="PLL (all tokens)",
                                    Loss=pnll / pn, PPL=math.exp(pnll / pn),
                                    BPB=pnll / math.log(2) / (TEST_BYTES * pfrac), Tokens=pn,
                                    Seconds=time.time() - t1))
        print(f"{stage}: {results[-1]['Objective']} | loss {results[-1]['Loss']:.4f} | ppl {results[-1]['PPL']:.2f}")
        if first:
            complexity_rows.append(complexity(model, cfg))
            first = False
        del model
        gc.collect(); torch.cuda.empty_cache()

res_df = pd.DataFrame(results)
cx_df = pd.DataFrame(complexity_rows)
res_df.to_csv(RESULTS_DIR / "test_perplexity.csv", index=False)
cx_df.to_csv(RESULTS_DIR / "model_complexity.csv", index=False)
print("\nSaved CSVs to", RESULTS_DIR.resolve())


=== Mamba-2 130M ===
tokens in stream 11,414,415 | blocks evaluated 2,000 (17.9% of the sampled text)


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 376.58it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


pre-CPT: causal LM | loss 3.3145 | ppl 27.51


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 332.66it/s]


post-CPT: causal LM | loss 2.5103 | ppl 12.31

=== GPT-2 124M ===
tokens in stream 11,391,632 | blocks evaluated 2,000 (18.0% of the sampled text)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 228.49it/s]


... (5 outputs left out)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 346.65it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


pre-CPT: PLL (all tokens) | loss 2.0122 | ppl 7.48


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 311.45it/s]


post-CPT: PLL (all tokens) | loss 1.0413 | ppl 2.83

Saved CSVs to C:\Users\HP\Desktop\2022IMT-061 Priya Keshri MTP\comparison_results


In [6]:
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("TEST-SET RESULTS (lower is better)")
display(res_df.round({"Loss": 4, "PPL": 2, "BPB": 4, "Seconds": 1}))

TEST-SET RESULTS (lower is better)


,Model,Stage,Objective,Loss,PPL,BPB,Tokens,Seconds
0,Mamba-2 130M,pre-CPT,causal LM,3.3145,27.51,1.0561,2046000,184.1
1,Mamba-2 130M,post-CPT,causal LM,2.5103,12.31,0.7998,2046000,181.7
2,GPT-2 124M,pre-CPT,causal LM,3.4891,32.76,1.1095,2046000,39.1
3,GPT-2 124M,post-CPT,causal LM,2.6198,13.73,0.8330,2046000,39.4
4,BERT-base-uncased,pre-CPT,MLM (15% masked),2.6098,13.60,NaN,152570,7.4
5,BERT-base-uncased,pre-CPT,PLL (all tokens),2.0122,7.48,0.6476,51000,150.9
6,BERT-base-uncased,post-CPT,MLM (15% masked),1.2992,3.67,NaN,152570,7.3
7,BERT-base-uncased,post-CPT,PLL (all tokens),1.0413,2.83,0.3351,51000,146.0


In [7]:
print("MODEL COMPLEXITY")
display(cx_df.set_index("Model").T.round(2))

MODEL COMPLEXITY


Model,Mamba-2 130M,GPT-2 124M,BERT-base-uncased
Params (M),128.99,124.44,109.51
Embedding params (M),38.62,38.60,23.44
Non-embedding params (M),90.37,85.84,86.07
Size fp32 (MiB),492.06,474.70,417.76
Size bf16 (MiB),246.03,237.35,208.88
"Size on disk, CPT ckpt (MiB)",492.08,474.71,417.79
Fwd GFLOPs @512 tok,170.72,136.16,121.24
GFLOPs/token @512,0.33,0.27,0.24
Fwd GFLOPs @1024 tok,341.51,291.65,NaN
GFLOPs/token @1024,0.33,0.28,NaN


## Notes on the complexity table
- **Params** are unique parameters (tied embeddings counted once). *Embedding* = input word-embedding matrix only.
- **Size on disk** is the saved `model.safetensors` of your CPT checkpoint (fp32).
- **Forward GFLOPs** come from `torch.utils.flop_counter` (2 x multiply-accumulates of matmul / conv / attention ops, incl. the LM or MLM head). Element-wise ops are not counted. BERT is capped at 512 tokens, so its 1024 column is empty.
- **Mamba-2 caveat:** without `mamba_ssm` / `causal_conv1d` installed, HF uses a PyTorch reference path and the counter measures that. If you install the fused Triton kernels, the counter will under-report their FLOPs.
- **CPT train FLOPs** ~ 3 x forward FLOPs per sequence x number of training sequences in one epoch (the usual forward + backward estimate). It does not include evaluation or the optimizer.